In [2]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# ===================== USER INPUT =====================
r_vlm   = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"
r_novlm = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_wo_VLM_stat.tif"

hc = 0.5  # depth threshold (m)

out_A_vlm   = r"D:\Phd Research\Final_Raster\A_100yr_VLM_hc0p5.tif"
out_A_novlm = r"D:\Phd Research\Final_Raster\A_100yr_noVLM_hc0p5.tif"
out_dA      = r"D:\Phd Research\Final_Raster\dA_100yr_VLM_minus_noVLM_hc0p5.tif"
# ======================================================


def read_float(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        prof = src.profile
        nodata = src.nodata
    # mask nodata and non-finite
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    arr = np.where(np.isfinite(arr), arr, np.nan)
    return arr, prof


def match_to_template(src_arr, src_prof, tmpl_prof):
    """Reproject/resample src_arr onto template grid if grids differ."""
    same = (
        src_prof["crs"] == tmpl_prof["crs"]
        and src_prof["transform"] == tmpl_prof["transform"]
        and src_prof["width"] == tmpl_prof["width"]
        and src_prof["height"] == tmpl_prof["height"]
    )
    if same:
        return src_arr

    dst = np.full((tmpl_prof["height"], tmpl_prof["width"]), np.nan, dtype="float32")
    reproject(
        source=src_arr,
        destination=dst,
        src_transform=src_prof["transform"],
        src_crs=src_prof["crs"],
        dst_transform=tmpl_prof["transform"],
        dst_crs=tmpl_prof["crs"],
        resampling=Resampling.bilinear,
        src_nodata=np.nan,
        dst_nodata=np.nan,
    )
    return dst


def write_int8(path, arr, base_prof, nodata_val=-128):
    prof = base_prof.copy()
    prof.update(dtype=rasterio.int8, nodata=nodata_val, count=1, compress="LZW")
    with rasterio.open(path, "w", **prof) as dst:
        dst.write(arr.astype(np.int8), 1)


# ---------- Read rasters ----------
vlm, prof_vlm = read_float(r_vlm)
novlm, prof_novlm = read_float(r_novlm)

# ---------- Ensure NOVLM matches VLM grid ----------
novlm = match_to_template(novlm, prof_novlm, prof_vlm)

# valid cells where both exist
valid = np.isfinite(vlm) & np.isfinite(novlm)

# ---------- Step 1: Binary activation Ai ----------
A_vlm_bin   = np.full(vlm.shape, -128, dtype=np.int8)
A_novlm_bin = np.full(vlm.shape, -128, dtype=np.int8)

A_vlm_bin[valid]   = (vlm[valid]   >= hc).astype(np.int8)
A_novlm_bin[valid] = (novlm[valid] >= hc).astype(np.int8)

# ---------- Step 2: ΔA = A_VLM - A_noVLM ----------
dA = np.full(vlm.shape, -128, dtype=np.int8)
dA[valid] = (A_vlm_bin[valid] - A_novlm_bin[valid]).astype(np.int8)  # {-1,0,+1}

# ---------- Step 3: AAR_VLM ----------
# cell area in m^2 (UTM meters)
t = prof_vlm["transform"]
cell_area_m2 = abs(t.a * t.e)

area_vlm_m2   = np.sum(A_vlm_bin[valid] == 1)   * cell_area_m2
area_novlm_m2 = np.sum(A_novlm_bin[valid] == 1) * cell_area_m2

AAR_VLM = (area_vlm_m2 / area_novlm_m2) if area_novlm_m2 > 0 else np.nan

gain_m2 = np.sum(dA[valid] ==  1) * cell_area_m2
loss_m2 = np.sum(dA[valid] == -1) * cell_area_m2

print(f"hc = {hc:.2f} m")
print(f"Flooded area (VLM)      = {area_vlm_m2/1e6:.3f} km^2")
print(f"Flooded area (no VLM)   = {area_novlm_m2/1e6:.3f} km^2")
print(f"AAR_VLM                 = {AAR_VLM:.4f}")
print(f"Gain (ΔA=+1) area       = {gain_m2/1e6:.3f} km^2")
print(f"Loss (ΔA=-1) area       = {loss_m2/1e6:.3f} km^2")

# ---------- Save outputs ----------
write_int8(out_A_vlm,   A_vlm_bin,   prof_vlm, -128)
write_int8(out_A_novlm, A_novlm_bin, prof_vlm, -128)
write_int8(out_dA,      dA,          prof_vlm, -128)

print("Saved:")
print("  ", out_A_vlm)
print("  ", out_A_novlm)
print("  ", out_dA)


hc = 0.50 m
Flooded area (VLM)      = 13641.200 km^2
Flooded area (no VLM)   = 13561.880 km^2
AAR_VLM                 = 1.0058
Gain (ΔA=+1) area       = 79.360 km^2
Loss (ΔA=-1) area       = 0.040 km^2
Saved:
   D:\Phd Research\Final_Raster\A_100yr_VLM_hc0p5.tif
   D:\Phd Research\Final_Raster\A_100yr_noVLM_hc0p5.tif
   D:\Phd Research\Final_Raster\dA_100yr_VLM_minus_noVLM_hc0p5.tif
